## Figure 8a — January-hindcast O3-minimum dates

Plot action: read the canonical processed product(s), apply display-only selection/reshaping, and render this one logical figure as PNG and PDF with one shared stem.


Inputs: canonical relationships/figure08a.csv and
verification/precursor_metrics.csv. Diagnostic notebook 05 already
applied the centered 5-day mean, searched March–April, applied the fixed
threshold from all 230 WACCM springs, and counted members in exactly
5-day bins. The plotting block draws the stored counts and uses the
stored January-member minimum dates only for the minimum/median/maximum
display lines; it does not recalculate ozone minima, thresholds,
classifications, or histogram counts.
Outputs: figure08a.png and PDF. This is the
user-accepted V1 five-day-box layout selected on 2026-08-24.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import xarray as xr

def discover_repository_root() -> Path:
    """Locate the cloned ``code`` directory from a notebook kernel."""

    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "analysis").is_dir() and (candidate / "figures").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Cannot locate the Paper 1 code directory. Run the notebook from the "
        "code directory or one of its notebook subdirectories."
    )


REPOSITORY_ROOT = discover_repository_root()
DEFAULT_DERIVED_ROOT = REPOSITORY_ROOT / "work"
DATA_ROOT = Path(os.environ.get("PAPER1_ARCHIVE_ROOT", str(REPOSITORY_ROOT / "data"))).expanduser().resolve()
INPUT_DESTINATIONS = tuple(
    path.resolve()
    for path in (
        DATA_ROOT,
        DATA_ROOT / "B2000WCN001002_timefixed",
        DATA_ROOT / "BWCN",
        DATA_ROOT / "Hindcast",
        DATA_ROOT / "WACCM" / "march_hindcast",
        DATA_ROOT / "MERRA2M2I6NPANA",
        DATA_ROOT / "MERRA2_Processed",
        DATA_ROOT / "MLS",
        DATA_ROOT / "CO2x1SmidEmin_yBWCN_timefixed",
    )
)
PRODUCT_VERSION = "Paper1_828_repro_v1"


def is_within(candidate: Path, parent: Path) -> bool:
    return candidate == parent or parent in candidate.parents


def validate_staging_root(candidate: Path) -> Path:
    root = candidate.expanduser().resolve()
    if root == Path(root.anchor) or root == DATA_ROOT:
        raise PermissionError(f"Refusing unsafe staging root: {root}")
    if root == Path(root.anchor):
        raise PermissionError("PAPER1_DERIVED_ROOT cannot be a filesystem root")
    for protected in INPUT_DESTINATIONS:
        if is_within(root, protected) or is_within(protected, root):
            raise PermissionError(
                "Refusing staging root that overlaps protected raw/legacy "
                f"tree: root={root}, protected={protected}"
            )
    return root


def canonical_root() -> Path:
    # Return the dedicated, ignored work tree (or an explicit safe child).
    override = os.environ.get("PAPER1_DERIVED_ROOT")
    if override:
        return validate_staging_root(Path(override))
    return validate_staging_root(DEFAULT_DERIVED_ROOT)


DERIVED_ROOT = canonical_root()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if not is_within(OUTPUT_DIR, DERIVED_ROOT):
    raise PermissionError(
        f"PAPER1_FIGURE_ROOT must remain below PAPER1_DERIVED_ROOT: {OUTPUT_DIR}"
    )


def canonical_path(relative: str) -> Path:
    path = DERIVED_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing canonical Paper 1 product: {path}. "
            "Run the analysis notebooks or set PAPER1_DERIVED_ROOT."
        )
    return path


def load_dataset(
    relative: str,
    required_variables: tuple[str, ...],
    *,
    require_version: bool = True,
) -> xr.Dataset:
    path = canonical_path(relative)
    with xr.open_dataset(path, decode_times=False) as opened:
        dataset = opened.load()
    missing = [name for name in required_variables if name not in dataset]
    if missing:
        raise ValueError(f"{path} is missing canonical variables {missing}")
    if require_version and dataset.attrs.get("product_version") != PRODUCT_VERSION:
        raise ValueError(
            f"{path}: product_version={dataset.attrs.get('product_version')!r}; "
            f"expected {PRODUCT_VERSION!r}"
        )
    return dataset


def require_columns(
    frame: pd.DataFrame, path: Path, columns: tuple[str, ...]
) -> None:
    missing = [name for name in columns if name not in frame]
    if missing:
        raise ValueError(f"{path} is missing canonical columns {missing}")
    if "product_version" not in frame:
        raise ValueError(f"{path} is missing product_version")
    versions = set(frame["product_version"].dropna().astype(str))
    if versions != {PRODUCT_VERSION}:
        raise ValueError(
            f"{path}: product_version values {sorted(versions)!r}; "
            f"expected only {PRODUCT_VERSION!r}"
        )


def canonical_waccm_threshold() -> float:
    # Read, but never reconstruct, the fixed low-25 threshold from 230 springs.
    path = canonical_path("ozone/waccm_master_rankings.csv")
    ranking = pd.read_csv(path)
    require_columns(
        ranking, path,
        (
            "sample_size", "low25_count", "low25_threshold_du",
            "is_low25",
        ),
    )
    if len(ranking) != 230:
        raise ValueError(f"{path}: expected exactly 230 ranked springs")
    if set(ranking["sample_size"].astype(int)) != {230}:
        raise ValueError(f"{path}: sample_size must be 230 on every row")
    if set(ranking["low25_count"].astype(int)) != {57}:
        raise ValueError(f"{path}: fixed low25 count must be floor(230/4)=57")
    thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if len(thresholds) != 1 or not np.isfinite(thresholds[0]):
        raise ValueError(f"{path}: expected one finite fixed threshold")
    return float(thresholds[0])


def validate_fixed_threshold(values: pd.Series, path: Path) -> float:
    stored = pd.to_numeric(values, errors="raise").unique()
    if len(stored) != 1 or not np.isfinite(stored[0]):
        raise ValueError(f"{path}: expected one finite stored low25 threshold")
    master = canonical_waccm_threshold()
    if not np.isclose(float(stored[0]), master, rtol=0.0, atol=1e-10):
        raise ValueError(
            f"{path}: threshold {stored[0]} differs from 230-spring "
            f"master threshold {master}"
        )
    return master


def parse_boolean(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(int).astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = values.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(
            f"Cannot parse boolean values {values[parsed.isna()].unique()}"
        )
    return parsed.astype(bool)


def text_value(value: object) -> str:
    # Decode NetCDF byte-string coordinates without producing "b'...'" labels.
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8")
    return str(value)


def date_parts(
    date_variable: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = np.asarray(date_variable.values)
    if values.ndim != 1:
        raise ValueError(f"date must be one-dimensional, found {values.shape}")
    if np.issubdtype(values.dtype, np.datetime64):
        dates = pd.DatetimeIndex(values)
        return (
            dates.year.to_numpy(dtype=int),
            dates.month.to_numpy(dtype=int),
            dates.day.to_numpy(dtype=int),
        )
    compact = values.astype(np.int64)
    return compact // 10000, (compact % 10000) // 100, compact % 100


def pressure_slice(
    values: xr.DataArray, pressure_hpa: float
) -> xr.DataArray:
    if "plev" not in values.dims:
        raise ValueError(
            f"Expected canonical plev dimension, found {values.dims}"
        )
    levels = np.asarray(values["plev"].values, dtype=float)
    if np.nanmax(levels) > 1100.0:
        raise ValueError("Canonical plev must be stored in hPa")
    selected = values.sel(plev=float(pressure_hpa), method="nearest")
    actual = float(selected["plev"])
    tolerance = max(0.6, pressure_hpa * 0.02)
    if not np.isclose(actual, pressure_hpa, rtol=0.0, atol=tolerance):
        raise ValueError(
            f"Requested {pressure_hpa} hPa; nearest level is {actual}"
        )
    return selected


def save_figure(
    figure: plt.Figure, stem: str, *, dpi: int = 300
) -> None:
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Figure stem must be one safe filename: {stem!r}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        figure.savefig(
            temporary_png, format="png", dpi=dpi,
            bbox_inches="tight", facecolor="white",
        )
        figure.savefig(
            temporary_pdf, format="pdf", bbox_inches="tight",
            facecolor="white",
        )
        if temporary_png.stat().st_size < 1024 or temporary_pdf.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {stem}")
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        plt.close(figure)
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved {png}")
    print(f"saved {pdf}")
bin_path = canonical_path("relationships/figure08a.csv")
metric_path = canonical_path("verification/precursor_metrics.csv")
bins = pd.read_csv(bin_path)
all_metrics = pd.read_csv(metric_path)
columns = (
    "bin_left_doy", "bin_right_doy",
    "count_all", "count_low", "count_other",
    "low25_threshold_du", "minimum_method",
)
require_columns(bins, bin_path, columns)
require_columns(
    all_metrics, metric_path,
    ("case", "member", "minimum_doy", "is_low",
     "low25_threshold_du", "minimum_method"),
)
threshold_du = validate_fixed_threshold(bins["low25_threshold_du"], bin_path)
validate_fixed_threshold(all_metrics["low25_threshold_du"], metric_path)
metrics = all_metrics.loc[all_metrics["case"].astype(str).eq("0008-01")].copy()
metrics["is_low"] = parse_boolean(metrics["is_low"])
if len(metrics) != 30 or metrics["member"].astype(str).nunique() != 30:
    raise ValueError(f"{metric_path}: expected 30 unique January members")
minimum_methods = bins["minimum_method"].astype(str).unique()
if len(minimum_methods) != 1 or "centered5" not in minimum_methods[0].lower():
    raise ValueError(f"{bin_path}: expected centered5 minimum metadata")
metric_methods = metrics["minimum_method"].astype(str).unique()
if len(metric_methods) != 1 or "centered5" not in metric_methods[0].lower():
    raise ValueError(f"{metric_path}: expected centered5 minimum metadata")
if not np.array_equal(
    bins["count_all"].to_numpy(dtype=int),
    bins["count_low"].to_numpy(dtype=int)
    + bins["count_other"].to_numpy(dtype=int),
):
    raise ValueError("Figure 8a stored bin counts are inconsistent")
if (
    int(bins["count_all"].sum()) != 30
    or int(bins["count_low"].sum()) != int(metrics["is_low"].sum())
    or int(bins["count_other"].sum()) != int((~metrics["is_low"]).sum())
):
    raise ValueError("Figure 8a stored bins disagree with January member metrics")
left = bins["bin_left_doy"].to_numpy(dtype=float)
right = bins["bin_right_doy"].to_numpy(dtype=float)
plot_left = left - 0.5
plot_right = np.minimum(right, 121.0) - 0.5
centers = (plot_left + plot_right) / 2.0
widths = (plot_right - plot_left) * 0.90

def doy_label(doy: int) -> str:
    date = pd.Timestamp("2001-01-01") + pd.Timedelta(days=int(doy) - 1)
    return f"{date.strftime('%b')} {date.day}"

labels = []
for first, stop in zip(left.astype(int), np.minimum(right, 121).astype(int)):
    last = stop - 1
    labels.append(doy_label(first) if first == last else f"{doy_label(first)}–{doy_label(last)}")

blue, orange = "#2f7fbd", "#e6862f"
figure, axis = plt.subplots(figsize=(11.8, 6.3), constrained_layout=True)
axis.bar(
    centers, bins["count_other"], width=widths,
    color=blue, edgecolor="white", alpha=0.88,
    label=f"Other members (n={int((~metrics['is_low']).sum())})",
)
axis.bar(
    centers, bins["count_low"], width=widths,
    bottom=bins["count_other"], color=orange, edgecolor="white", alpha=0.90,
    label=(f"Fixed WACCM low-25%: O$_3$ min ≤ {threshold_du:.1f} DU "
           f"(n={int(metrics['is_low'].sum())})"),
)
minimum = int(metrics["minimum_doy"].min())
median = float(metrics["minimum_doy"].median())
maximum = int(metrics["minimum_doy"].max())
axis.axvline(minimum, color="0.25", lw=1.1, ls=":", label=f"range {doy_label(minimum)}–{doy_label(maximum)}")
axis.axvline(median, color=blue, lw=1.6, ls="--", label=f"median {doy_label(round(median))}")
axis.axvline(maximum, color="0.25", lw=1.1, ls=":")
axis.set_xlim(59.5, 120.5)
axis.set_xticks(centers, labels, rotation=35, ha="right")
axis.set_ylim(bottom=0)
axis.yaxis.set_major_locator(MaxNLocator(integer=True))
axis.set_xlabel("Date of centered-5-day March–April O$_3$ minimum")
axis.set_ylabel("Member count per 5-day box")
axis.set_title("January initialization: dates of March–April O$_3$ minima (5-day boxes)")
axis.grid(True, axis="y", ls="--", alpha=0.25)
axis.legend(frameon=True, loc="upper left")
save_figure(figure, "figure08a")


## Figure 8b — January–February EP100 versus O3 minimum

Plot action: read the canonical processed product(s), apply display-only selection/reshaping, and render this one logical figure as PNG and PDF with one shared stem.


Inputs: canonical relationships/figure08b.csv. It stores the 30 member
points, fixed low-25% flags, January–February no-W/monthly-N2 EP100
window means, centered-5-day March–April O3 minima, WACCM year-0008
reference location, and precomputed Pearson/OLS statistics. The
plotting block only renders those stored values.

Outputs: figure08b_jan_wave_vs_o3minimum_raw.png and PDF.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

def discover_repository_root() -> Path:
    """Locate the cloned ``code`` directory from a notebook kernel."""

    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "analysis").is_dir() and (candidate / "figures").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Cannot locate the Paper 1 code directory. Run the notebook from the "
        "code directory or one of its notebook subdirectories."
    )


REPOSITORY_ROOT = discover_repository_root()
DEFAULT_DERIVED_ROOT = REPOSITORY_ROOT / "work"
DATA_ROOT = Path(os.environ.get("PAPER1_ARCHIVE_ROOT", str(REPOSITORY_ROOT / "data"))).expanduser().resolve()
INPUT_DESTINATIONS = tuple(
    path.resolve()
    for path in (
        DATA_ROOT,
        DATA_ROOT / "B2000WCN001002_timefixed",
        DATA_ROOT / "BWCN",
        DATA_ROOT / "Hindcast",
        DATA_ROOT / "WACCM" / "march_hindcast",
        DATA_ROOT / "MERRA2M2I6NPANA",
        DATA_ROOT / "MERRA2_Processed",
        DATA_ROOT / "MLS",
        DATA_ROOT / "CO2x1SmidEmin_yBWCN_timefixed",
    )
)
PRODUCT_VERSION = "Paper1_828_repro_v1"


def is_within(candidate: Path, parent: Path) -> bool:
    return candidate == parent or parent in candidate.parents


def validate_staging_root(candidate: Path) -> Path:
    root = candidate.expanduser().resolve()
    if root == Path(root.anchor) or root == DATA_ROOT:
        raise PermissionError(f"Refusing unsafe staging root: {root}")
    if root == Path(root.anchor):
        raise PermissionError("PAPER1_DERIVED_ROOT cannot be a filesystem root")
    for protected in INPUT_DESTINATIONS:
        if is_within(root, protected) or is_within(protected, root):
            raise PermissionError(
                "Refusing staging root that overlaps protected raw/legacy "
                f"tree: root={root}, protected={protected}"
            )
    return root


def canonical_root() -> Path:
    # Return the dedicated, ignored work tree (or an explicit safe child).
    override = os.environ.get("PAPER1_DERIVED_ROOT")
    if override:
        return validate_staging_root(Path(override))
    return validate_staging_root(DEFAULT_DERIVED_ROOT)


DERIVED_ROOT = canonical_root()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if not is_within(OUTPUT_DIR, DERIVED_ROOT):
    raise PermissionError(
        f"PAPER1_FIGURE_ROOT must remain below PAPER1_DERIVED_ROOT: {OUTPUT_DIR}"
    )


def canonical_path(relative: str) -> Path:
    path = DERIVED_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing canonical Paper 1 product: {path}. "
            "Run the analysis notebooks or set PAPER1_DERIVED_ROOT."
        )
    return path


def load_dataset(
    relative: str,
    required_variables: tuple[str, ...],
    *,
    require_version: bool = True,
) -> xr.Dataset:
    path = canonical_path(relative)
    with xr.open_dataset(path, decode_times=False) as opened:
        dataset = opened.load()
    missing = [name for name in required_variables if name not in dataset]
    if missing:
        raise ValueError(f"{path} is missing canonical variables {missing}")
    if require_version and dataset.attrs.get("product_version") != PRODUCT_VERSION:
        raise ValueError(
            f"{path}: product_version={dataset.attrs.get('product_version')!r}; "
            f"expected {PRODUCT_VERSION!r}"
        )
    return dataset


def require_columns(
    frame: pd.DataFrame, path: Path, columns: tuple[str, ...]
) -> None:
    missing = [name for name in columns if name not in frame]
    if missing:
        raise ValueError(f"{path} is missing canonical columns {missing}")
    if "product_version" not in frame:
        raise ValueError(f"{path} is missing product_version")
    versions = set(frame["product_version"].dropna().astype(str))
    if versions != {PRODUCT_VERSION}:
        raise ValueError(
            f"{path}: product_version values {sorted(versions)!r}; "
            f"expected only {PRODUCT_VERSION!r}"
        )


def canonical_waccm_threshold() -> float:
    # Read, but never reconstruct, the fixed low-25 threshold from 230 springs.
    path = canonical_path("ozone/waccm_master_rankings.csv")
    ranking = pd.read_csv(path)
    require_columns(
        ranking, path,
        (
            "sample_size", "low25_count", "low25_threshold_du",
            "is_low25",
        ),
    )
    if len(ranking) != 230:
        raise ValueError(f"{path}: expected exactly 230 ranked springs")
    if set(ranking["sample_size"].astype(int)) != {230}:
        raise ValueError(f"{path}: sample_size must be 230 on every row")
    if set(ranking["low25_count"].astype(int)) != {57}:
        raise ValueError(f"{path}: fixed low25 count must be floor(230/4)=57")
    thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if len(thresholds) != 1 or not np.isfinite(thresholds[0]):
        raise ValueError(f"{path}: expected one finite fixed threshold")
    return float(thresholds[0])


def validate_fixed_threshold(values: pd.Series, path: Path) -> float:
    stored = pd.to_numeric(values, errors="raise").unique()
    if len(stored) != 1 or not np.isfinite(stored[0]):
        raise ValueError(f"{path}: expected one finite stored low25 threshold")
    master = canonical_waccm_threshold()
    if not np.isclose(float(stored[0]), master, rtol=0.0, atol=1e-10):
        raise ValueError(
            f"{path}: threshold {stored[0]} differs from 230-spring "
            f"master threshold {master}"
        )
    return master


def parse_boolean(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(int).astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = values.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(
            f"Cannot parse boolean values {values[parsed.isna()].unique()}"
        )
    return parsed.astype(bool)


def text_value(value: object) -> str:
    # Decode NetCDF byte-string coordinates without producing "b'...'" labels.
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8")
    return str(value)


def date_parts(
    date_variable: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = np.asarray(date_variable.values)
    if values.ndim != 1:
        raise ValueError(f"date must be one-dimensional, found {values.shape}")
    if np.issubdtype(values.dtype, np.datetime64):
        dates = pd.DatetimeIndex(values)
        return (
            dates.year.to_numpy(dtype=int),
            dates.month.to_numpy(dtype=int),
            dates.day.to_numpy(dtype=int),
        )
    compact = values.astype(np.int64)
    return compact // 10000, (compact % 10000) // 100, compact % 100


def pressure_slice(
    values: xr.DataArray, pressure_hpa: float
) -> xr.DataArray:
    if "plev" not in values.dims:
        raise ValueError(
            f"Expected canonical plev dimension, found {values.dims}"
        )
    levels = np.asarray(values["plev"].values, dtype=float)
    if np.nanmax(levels) > 1100.0:
        raise ValueError("Canonical plev must be stored in hPa")
    selected = values.sel(plev=float(pressure_hpa), method="nearest")
    actual = float(selected["plev"])
    tolerance = max(0.6, pressure_hpa * 0.02)
    if not np.isclose(actual, pressure_hpa, rtol=0.0, atol=tolerance):
        raise ValueError(
            f"Requested {pressure_hpa} hPa; nearest level is {actual}"
        )
    return selected


def save_figure(
    figure: plt.Figure, stem: str, *, dpi: int = 300
) -> None:
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Figure stem must be one safe filename: {stem!r}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        figure.savefig(
            temporary_png, format="png", dpi=dpi,
            bbox_inches="tight", facecolor="white",
        )
        figure.savefig(
            temporary_pdf, format="pdf", bbox_inches="tight",
            facecolor="white",
        )
        if temporary_png.stat().st_size < 1024 or temporary_pdf.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {stem}")
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        plt.close(figure)
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved {png}")
    print(f"saved {pdf}")
def draw_stored_member_relationship(
    relative: str,
    stem: str,
    title: str,
    x_label: str,
    y_label: str,
) -> None:
    path = canonical_path(relative)
    table = pd.read_csv(path)
    required = (
        "case", "member", "x", "y", "is_low", "r", "p", "n",
        "slope", "intercept", "ref_x", "ref_y",
        "low25_threshold_du", "epflux_method", "minimum_method",
    )
    require_columns(table, path, required)
    table["is_low"] = parse_boolean(table["is_low"])
    for name in ("r", "p", "n", "slope", "intercept", "ref_x", "ref_y"):
        finite_unique = pd.to_numeric(table[name], errors="raise").dropna().unique()
        if len(finite_unique) != 1:
            raise ValueError(
                f"{path}: repeated metadata column {name} must have "
                "one finite value"
            )
    validate_fixed_threshold(table["low25_threshold_du"], path)
    ep_methods = table["epflux_method"].astype(str).unique()
    minimum_methods = table["minimum_method"].astype(str).unique()
    if len(ep_methods) != 1 or len(minimum_methods) != 1:
        raise ValueError(f"{path}: method metadata is not unique")
    ep_method = ep_methods[0].lower()
    for token_group in (
        ("ubar",), ("monthly",),
        ("w=none", "w = none", "no omega"),
        ("40--80", "40–80"),
    ):
        if not any(token in ep_method for token in token_group):
            raise ValueError(f"{path}: epflux_method fails {token_group}")
    minimum_method = minimum_methods[0].lower()
    for token in ("centered5", "mar1-apr30", "30--70", "60--90"):
        if token not in minimum_method:
            raise ValueError(f"{path}: minimum_method lacks {token!r}")
    metadata = table.iloc[0]
    low = table["is_low"].astype(bool)
    figure, axis = plt.subplots(figsize=(7.5, 6.3))
    axis.scatter(
        table.loc[~low, "x"], table.loc[~low, "y"],
        s=48, color="#4c78a8", edgecolor="white", linewidth=0.45,
        label="other members",
    )
    axis.scatter(
        table.loc[low, "x"], table.loc[low, "y"],
        s=48, color="#e6550d", edgecolor="white", linewidth=0.45,
        label="fixed low25 members",
    )
    for _, row in table.iterrows():
        axis.annotate(
            str(row["member"]), (row["x"], row["y"]),
            xytext=(3, 3), textcoords="offset points",
            fontsize=6.4, color="0.25",
        )
    x_grid = np.linspace(table["x"].min(), table["x"].max(), 100)
    axis.plot(
        x_grid,
        float(metadata["intercept"]) + float(metadata["slope"]) * x_grid,
        color="0.12", lw=1.5,
    )
    if np.isfinite(float(metadata["ref_x"])) and np.isfinite(float(metadata["ref_y"])):
        axis.scatter(
            float(metadata["ref_x"]), float(metadata["ref_y"]),
            marker="*", s=150, color="#ffd92f", edgecolor="black",
            linewidth=0.8, zorder=7, label="WACCM year 0008",
        )
    p_value = float(metadata["p"])
    p_label = "<0.001" if p_value < 0.001 else f"={p_value:.3f}"
    axis.text(
        0.04, 0.96,
        f"Stored Pearson r={float(metadata['r']):.2f}, "
        f"p{p_label}, N={int(metadata['n'])}",
        transform=axis.transAxes, ha="left", va="top",
        bbox={"facecolor": "white", "edgecolor": "0.82", "alpha": 0.9},
    )
    axis.set_xlabel(x_label)
    axis.set_ylabel(y_label)
    axis.set_title(title, fontweight="bold")
    axis.grid(color="0.88", lw=0.5)
    axis.legend(frameon=False)
    save_figure(figure, stem)
draw_stored_member_relationship(
    "relationships/figure08b.csv",
    "figure08b_jan_wave_vs_o3minimum_raw",
    "January initialization: Jan–Feb EP100 versus spring O$_3$ minimum",
    "Stored Jan–Feb upward EP100 mean",
    "Stored March–April O$_3$ minimum (DU)",
)


## Figure 8h — Jan 21–Feb 9 EP100 versus whole-period O3 RMSE

Plot action: read the canonical processed table and render member groups,
stored Pearson/OLS statistics, and the no-W year-0008 EP100 reference line.

Inputs: `relationships/figure08h.csv`. Diagnostic notebook 05 stores exactly
30 January members, their 20-day (DOY 21–40) no-W/monthly-N² EP100 means, and
their Jan 1–May 30 partial-O3 RMSE values (Nt=150). The best and worst five are
display groups ranked only by the stored RMSE; the reference line is excluded
from the stored member regression.

Outputs: `figure08h.png` and PDF.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

def discover_repository_root() -> Path:
    """Locate the cloned ``code`` directory from a notebook kernel."""

    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "analysis").is_dir() and (candidate / "figures").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Cannot locate the Paper 1 code directory. Run the notebook from the "
        "code directory or one of its notebook subdirectories."
    )


REPOSITORY_ROOT = discover_repository_root()
DEFAULT_DERIVED_ROOT = REPOSITORY_ROOT / "work"
DATA_ROOT = Path(os.environ.get("PAPER1_ARCHIVE_ROOT", str(REPOSITORY_ROOT / "data"))).expanduser().resolve()
INPUT_DESTINATIONS = tuple(
    path.resolve()
    for path in (
        DATA_ROOT,
        DATA_ROOT / "B2000WCN001002_timefixed",
        DATA_ROOT / "BWCN",
        DATA_ROOT / "Hindcast",
        DATA_ROOT / "WACCM" / "march_hindcast",
        DATA_ROOT / "MERRA2M2I6NPANA",
        DATA_ROOT / "MERRA2_Processed",
        DATA_ROOT / "MLS",
        DATA_ROOT / "CO2x1SmidEmin_yBWCN_timefixed",
    )
)
PRODUCT_VERSION = "Paper1_828_repro_v1"


def is_within(candidate: Path, parent: Path) -> bool:
    return candidate == parent or parent in candidate.parents


def validate_staging_root(candidate: Path) -> Path:
    root = candidate.expanduser().resolve()
    if root == Path(root.anchor) or root == DATA_ROOT:
        raise PermissionError(f"Refusing unsafe staging root: {root}")
    if root == Path(root.anchor):
        raise PermissionError("PAPER1_DERIVED_ROOT cannot be a filesystem root")
    for protected in INPUT_DESTINATIONS:
        if is_within(root, protected) or is_within(protected, root):
            raise PermissionError(
                "Refusing staging root that overlaps protected raw/legacy "
                f"tree: root={root}, protected={protected}"
            )
    return root


def canonical_root() -> Path:
    # Return the dedicated, ignored work tree (or an explicit safe child).
    override = os.environ.get("PAPER1_DERIVED_ROOT")
    if override:
        return validate_staging_root(Path(override))
    return validate_staging_root(DEFAULT_DERIVED_ROOT)


DERIVED_ROOT = canonical_root()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if not is_within(OUTPUT_DIR, DERIVED_ROOT):
    raise PermissionError(
        f"PAPER1_FIGURE_ROOT must remain below PAPER1_DERIVED_ROOT: {OUTPUT_DIR}"
    )


def canonical_path(relative: str) -> Path:
    path = DERIVED_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing canonical Paper 1 product: {path}. "
            "Run the analysis notebooks or set PAPER1_DERIVED_ROOT."
        )
    return path


def load_dataset(
    relative: str,
    required_variables: tuple[str, ...],
    *,
    require_version: bool = True,
) -> xr.Dataset:
    path = canonical_path(relative)
    with xr.open_dataset(path, decode_times=False) as opened:
        dataset = opened.load()
    missing = [name for name in required_variables if name not in dataset]
    if missing:
        raise ValueError(f"{path} is missing canonical variables {missing}")
    if require_version and dataset.attrs.get("product_version") != PRODUCT_VERSION:
        raise ValueError(
            f"{path}: product_version={dataset.attrs.get('product_version')!r}; "
            f"expected {PRODUCT_VERSION!r}"
        )
    return dataset


def require_columns(
    frame: pd.DataFrame, path: Path, columns: tuple[str, ...]
) -> None:
    missing = [name for name in columns if name not in frame]
    if missing:
        raise ValueError(f"{path} is missing canonical columns {missing}")
    if "product_version" not in frame:
        raise ValueError(f"{path} is missing product_version")
    versions = set(frame["product_version"].dropna().astype(str))
    if versions != {PRODUCT_VERSION}:
        raise ValueError(
            f"{path}: product_version values {sorted(versions)!r}; "
            f"expected only {PRODUCT_VERSION!r}"
        )


def canonical_waccm_threshold() -> float:
    # Read, but never reconstruct, the fixed low-25 threshold from 230 springs.
    path = canonical_path("ozone/waccm_master_rankings.csv")
    ranking = pd.read_csv(path)
    require_columns(
        ranking, path,
        (
            "sample_size", "low25_count", "low25_threshold_du",
            "is_low25",
        ),
    )
    if len(ranking) != 230:
        raise ValueError(f"{path}: expected exactly 230 ranked springs")
    if set(ranking["sample_size"].astype(int)) != {230}:
        raise ValueError(f"{path}: sample_size must be 230 on every row")
    if set(ranking["low25_count"].astype(int)) != {57}:
        raise ValueError(f"{path}: fixed low25 count must be floor(230/4)=57")
    thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if len(thresholds) != 1 or not np.isfinite(thresholds[0]):
        raise ValueError(f"{path}: expected one finite fixed threshold")
    return float(thresholds[0])


def validate_fixed_threshold(values: pd.Series, path: Path) -> float:
    stored = pd.to_numeric(values, errors="raise").unique()
    if len(stored) != 1 or not np.isfinite(stored[0]):
        raise ValueError(f"{path}: expected one finite stored low25 threshold")
    master = canonical_waccm_threshold()
    if not np.isclose(float(stored[0]), master, rtol=0.0, atol=1e-10):
        raise ValueError(
            f"{path}: threshold {stored[0]} differs from 230-spring "
            f"master threshold {master}"
        )
    return master


def parse_boolean(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(int).astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = values.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(
            f"Cannot parse boolean values {values[parsed.isna()].unique()}"
        )
    return parsed.astype(bool)


def text_value(value: object) -> str:
    # Decode NetCDF byte-string coordinates without producing "b'...'" labels.
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8")
    return str(value)


def date_parts(
    date_variable: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = np.asarray(date_variable.values)
    if values.ndim != 1:
        raise ValueError(f"date must be one-dimensional, found {values.shape}")
    if np.issubdtype(values.dtype, np.datetime64):
        dates = pd.DatetimeIndex(values)
        return (
            dates.year.to_numpy(dtype=int),
            dates.month.to_numpy(dtype=int),
            dates.day.to_numpy(dtype=int),
        )
    compact = values.astype(np.int64)
    return compact // 10000, (compact % 10000) // 100, compact % 100


def pressure_slice(
    values: xr.DataArray, pressure_hpa: float
) -> xr.DataArray:
    if "plev" not in values.dims:
        raise ValueError(
            f"Expected canonical plev dimension, found {values.dims}"
        )
    levels = np.asarray(values["plev"].values, dtype=float)
    if np.nanmax(levels) > 1100.0:
        raise ValueError("Canonical plev must be stored in hPa")
    selected = values.sel(plev=float(pressure_hpa), method="nearest")
    actual = float(selected["plev"])
    tolerance = max(0.6, pressure_hpa * 0.02)
    if not np.isclose(actual, pressure_hpa, rtol=0.0, atol=tolerance):
        raise ValueError(
            f"Requested {pressure_hpa} hPa; nearest level is {actual}"
        )
    return selected


def save_figure(
    figure: plt.Figure, stem: str, *, dpi: int = 300
) -> None:
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Figure stem must be one safe filename: {stem!r}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        figure.savefig(
            temporary_png, format="png", dpi=dpi,
            bbox_inches="tight", facecolor="white",
        )
        figure.savefig(
            temporary_pdf, format="pdf", bbox_inches="tight",
            facecolor="white",
        )
        if temporary_png.stat().st_size < 1024 or temporary_pdf.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {stem}")
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        plt.close(figure)
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved {png}")
    print(f"saved {pdf}")
path = canonical_path("relationships/figure08h.csv")
table = pd.read_csv(path)
required = (
    "case", "member", "x", "y", "r", "p", "n", "slope", "intercept",
    "ref_x", "window_start_doy", "window_end_doy", "window_days",
    "evaluation_start", "evaluation_end", "evaluation_nt",
    "epflux_method", "rmse_method",
)
require_columns(table, path, required)
if len(table) != 30 or table["member"].astype(str).nunique() != 30:
    raise ValueError(f"{path}: expected 30 unique January members")
for name, expected in {
    "window_start_doy": 21, "window_end_doy": 40,
    "window_days": 20, "evaluation_nt": 150,
}.items():
    if set(pd.to_numeric(table[name], errors="raise").astype(int)) != {expected}:
        raise ValueError(f"{path}: {name} must be {expected}")
for name in ("r", "p", "n", "slope", "intercept", "ref_x"):
    values = pd.to_numeric(table[name], errors="raise").dropna().unique()
    if len(values) != 1:
        raise ValueError(f"{path}: {name} must contain one stored value")
ep_method = table["epflux_method"].astype(str).iloc[0].lower()
for token in ("monthly", "w=none", "40--80"):
    if token not in ep_method:
        raise ValueError(f"{path}: EP-flux metadata lacks {token!r}")

metadata = table.iloc[0]
x = pd.to_numeric(table["x"], errors="raise").to_numpy(float)
y = pd.to_numeric(table["y"], errors="raise").to_numpy(float)
order = np.argsort(y, kind="stable")
best, worst = order[:5], order[-5:]
other = np.setdiff1d(np.arange(len(table)), np.union1d(best, worst))

figure, axis = plt.subplots(figsize=(7.7, 5.65), constrained_layout=True)
axis.scatter(x[other], y[other], s=55, color="0.25", alpha=0.72,
             edgecolor="white", linewidth=0.5, zorder=2, label="Other members")
axis.scatter(x[best], y[best], s=78, color="#2f7fbd", edgecolor="black",
             linewidth=0.55, zorder=4, label="Best 5: lowest O$_3$ RMSE")
axis.scatter(x[worst], y[worst], s=78, color="#c8323e", edgecolor="black",
             linewidth=0.55, zorder=4, label="Worst 5: highest O$_3$ RMSE")
reference_x = float(metadata["ref_x"])
x_all = np.r_[x, reference_x]
x_margin = max(float(np.ptp(x_all)) * 0.09, 0.05)
y_margin = max(float(np.ptp(y)) * 0.09, 0.5)
x_limits = (float(np.min(x_all) - x_margin), float(np.max(x_all) + x_margin))
x_line = np.linspace(*x_limits, 200)
axis.plot(
    x_line, float(metadata["intercept"]) + float(metadata["slope"]) * x_line,
    color="#b72f3b", lw=1.9, ls="--", zorder=3,
)
axis.axvline(reference_x, color="black", lw=1.7, zorder=3.5,
             label="BWCN year 0008 (no-W)")
p_value = float(metadata["p"])
p_label = "<0.001" if p_value < 0.001 else f"={p_value:.3f}"
axis.text(
    0.97, 0.97,
    f"Pearson r={float(metadata['r']):.2f}, p{p_label}\nn={int(metadata['n'])}",
    transform=axis.transAxes, ha="right", va="top", fontsize=9.5,
    bbox=dict(facecolor="white", edgecolor="0.72", alpha=0.9, pad=3),
)
axis.set_xlim(*x_limits)
axis.set_ylim(float(np.min(y) - y_margin), float(np.max(y) + y_margin))
axis.set_xlabel("Jan 21–Feb 9 mean upward EP100 ($10^{-3}$ hPa m s$^{-2}$; 40–80° N)")
axis.set_ylabel("Whole-period O$_3$ RMSE (Jan 1–May 30; DU)")
axis.set_title("January initialization: O$_3$ RMSE versus mid-winter wave activity")
axis.grid(True, ls="--", alpha=0.25)
axis.legend(loc="lower right", frameon=True, fontsize=8.5)
save_figure(figure, "figure08h")
